In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces

# ------------------------------
# 1. 定义环境与网络（与训练时完全一致）
# ------------------------------
# 环境类 SimplePacmanEnv 已在前面给出，此处省略（直接复制用户代码）
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
GAMMA = 0.99

class SimplePacmanEnv(gym.Env):
    """
    简化的吃豆人环境：
    - 网格大小 grid_size  15
    - 玩家从 (0,0) 开始
    - 随机生成约 60% 的豆子
    - 吃到豆子得 +1，吃完所有豆子得 +10，每步 -0.1
    - 状态: 2通道 (玩家位置, 豆子分布)，每个通道 grid_size x grid_size
    """
    def __init__(self, grid_size=15):
        super().__init__()
        self.grid_size = grid_size

        # 动作空间: 0上,1下,2左,3右
        self.action_space = spaces.Discrete(4)

        # 观察空间: 2通道网格，每个像素值0或1
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(2, grid_size, grid_size),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # 玩家起始位置
        self.player_pos = [0, 0]

        # 随机生成豆子（约60%）
        total_cells = self.grid_size * self.grid_size
        num_dots = int(total_cells * 0.6)
        rng = np.random.default_rng(seed)
        indices = rng.choice(total_cells, num_dots, replace=False)
        self.dot_grid = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        self.dot_grid.flat[indices] = True
        self.dot_grid[0, 0] = False  # 玩家起点无豆

        self.dots_left = np.sum(self.dot_grid)
        self.steps = 0
        return self._get_obs(), {}

    def _get_obs(self):
        # 玩家通道
        player = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        px, py = self.player_pos
        player[px, py] = 1.0

        # 豆子通道
        dot = self.dot_grid.astype(np.float32)

        return np.stack([player, dot], axis=0)

    def step(self, action):
        self.steps += 1
        reward = -0.05
        done = False

        # 移动玩家
        x, y = self.player_pos
        if action == 0:   # 上
            y = min(y + 1, self.grid_size - 1)
        elif action == 1: # 下
            y = max(y - 1, 0)
        elif action == 2: # 左
            x = max(x - 1, 0)
        elif action == 3: # 右
            x = min(x + 1, self.grid_size - 1)
        self.player_pos = [x, y]

        # 吃豆
        if self.dot_grid[x, y]:
            self.dot_grid[x, y] = False
            self.dots_left -= 1
            reward += 1

        # 检查是否吃完所有豆子
        if self.dots_left == 0:
            reward += 10
            done = True

        # 步数上限
        if self.steps >= 1000:
            done = True

        return self._get_obs(), reward, done, {}

    def render(self, mode='human'):
        # 简单文本渲染
        grid = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        # 标记豆子
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.dot_grid[i, j]:
                    grid[i][j] = 'o'
        # 标记玩家
        px, py = self.player_pos
        grid[px][py] = 'P'
        # 打印
        for row in reversed(grid):
            print(' '.join(row))
        print(f"Steps: {self.steps}, Dots left: {self.dots_left}")
# 网络定义
class NoisyLinear(nn.Module):
    def __init__(self, in_features, out_features, std_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.Tensor(out_features))
        self.bias_sigma = nn.Parameter(torch.Tensor(out_features))
        self.std_init = std_init
        self.reset_parameters()
        self.register_buffer("weight_epsilon", torch.Tensor(out_features, in_features))
        self.register_buffer("bias_epsilon", torch.Tensor(out_features))

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.std_init / math.sqrt(self.in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(self.std_init / math.sqrt(self.out_features))

    def forward(self, x):
        if self.training:
            self.weight_epsilon.normal_()
            self.bias_epsilon.normal_()
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)

class QNetworkDuelingCNN(nn.Module):
    def __init__(self, input_channels=2, grid_size=15, action_size=4):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, 3, padding=1)
        self.flatten_dim = grid_size * grid_size * 64
        self.fc1 = NoisyLinear(self.flatten_dim, 256)
        self.value_fc = NoisyLinear(256, 1)
        self.advantage_fc = NoisyLinear(256, action_size)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(-1, self.flatten_dim)
        x = F.relu(self.fc1(x))
        value = self.value_fc(x)
        advantage = self.advantage_fc(x)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

# DQN Agent（只保留策略网络用于推理，不加载记忆体）
class DQNAgent:
    def __init__(self, action_size, device, model_path=None, 
                 anti_cycle_enabled=True,        # 新增开关
                 anti_cycle_window=6, 
                 anti_cycle_threshold=2):
        self.action_size = action_size
        self.device = device
        self.policy_net = QNetworkDuelingCNN().to(device)
        if model_path is not None:
            self.policy_net.load_state_dict(torch.load(model_path, map_location=device))
        self.policy_net.eval()

        # 防循环参数
        self.anti_cycle_enabled = anti_cycle_enabled
        self.anti_cycle_window = anti_cycle_window
        self.anti_cycle_threshold = anti_cycle_threshold
        if anti_cycle_enabled:
            self.action_history = []
            self.cycle_count = 0

    def select_action_deterministic(self, state):
        self.policy_net.eval()
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
            best_action = q_values.argmax().item()
            if self.anti_cycle_enabled:
                # 需要次优动作用于干预
                top2 = q_values.topk(2)
                second_best = top2.indices[0, 1].item() if top2.values[0, 1] > -1e9 else best_action
            else:
                return best_action

        # 防循环逻辑（仅当 anti_cycle_enabled=True）
        chosen_action = best_action
        if len(self.action_history) >= self.anti_cycle_window:
            recent = self.action_history[-self.anti_cycle_window:]
            if self._is_oscillating(recent):
                self.cycle_count += 1
                if self.cycle_count >= self.anti_cycle_threshold:
                    chosen_action = second_best
                    self.cycle_count = 0
            else:
                self.cycle_count = 0

        self.action_history.append(chosen_action)
        if len(self.action_history) > self.anti_cycle_window:
            self.action_history.pop(0)
        return chosen_action


    def _is_oscillating(self, seq):
        """检测动作序列是否在往返振荡（如 [0,1,0,1] 或 [1,0,1,0]）"""
        if len(seq) < 4:
            return False
        opposites = {0:1, 1:0, 2:3, 3:2}
        for i in range(1, len(seq)):
            if opposites.get(seq[i-1]) != seq[i]:
                return False
        return True

    def select_action_with_noise(self, state):
        """带噪声的动作选择（训练模式，NoisyLinear 随机性），可选防循环"""
        self.policy_net.train()
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
            best_action = q_values.argmax().item()
            if self.anti_cycle_enabled:
                top2 = q_values.topk(2)
                second_best = top2.indices[0, 1].item() if top2.values[0, 1] > -1e9 else best_action
            else:
                return best_action

        chosen_action = best_action
        if len(self.action_history) >= self.anti_cycle_window:
            recent = self.action_history[-self.anti_cycle_window:]
            if self._is_oscillating(recent):
                self.cycle_count += 1
                if self.cycle_count >= self.anti_cycle_threshold:
                    chosen_action = second_best
                    self.cycle_count = 0
            else:
                self.cycle_count = 0

        self.action_history.append(chosen_action)
        if len(self.action_history) > self.anti_cycle_window:
            self.action_history.pop(0)
        return chosen_action


# ------------------------------
# 2. 演示函数
# ------------------------------
def play_episode(env, agent, action_selector, render=True):
    """
    运行一个回合
    :param env: 环境实例
    :param agent: DQNAgent 实例
    :param action_selector: 函数，接受状态返回动作
    :param render: 是否每步打印渲染
    :return: (总奖励, 步数, 吃豆数)
    """
    state, _ = env.reset()
    initial_dots = env.dots_left
    total_reward = 0
    steps = 0
    done = False
    
    if render:
        print("\n初始状态：")
        env.render()

    while not done:
        action = action_selector(state)
        next_state, reward, done, _ = env.step(action)
        total_reward += reward
        steps += 1
        state = next_state

        if render:
            print(f"\n步骤 {steps}，动作 {action}，奖励 {reward:.2f}，总奖励 {total_reward:.2f}")
            env.render()

    dots_eaten = initial_dots - env.dots_left
    return total_reward, steps, dots_eaten

def demonstrate(agent, env, episodes, with_noise=False, with_anti_cycle=True, render=True):
    """
    演示 agent 玩游戏
    :param agent: DQNAgent 实例
    :param env: SimplePacmanEnv 实例
    :param episodes: 演示回合数
    :param with_noise: True 使用带噪声的动作选择（NoisyLinear 随机性），False 使用确定性动作选择
    :param with_anti_cycle: 是否启用防循环机制（仅在 with_noise=False 或已修改 select_action_with_noise 时生效）
    :param render: 是否每步打印环境状态
    """
    # 统一设置防循环标志（使得确定性模式和噪声模式都能根据此标志决定是否防循环）
    agent.anti_cycle_enabled = with_anti_cycle

    # 如果启用防循环但 agent 尚未初始化历史记录，则创建（某些代码可能只在 __init__ 中创建）
    if with_anti_cycle and not hasattr(agent, 'action_history'):
        agent.action_history = []
        agent.cycle_count = 0

    mode_name = "带噪声" if with_noise else "确定性"
    if with_anti_cycle:
        mode_name += " + 防循环"
    else:
        mode_name += "（无防循环）"

    print(f"\n========== 演示模式：{mode_name} ==========\n")

    total_reward = 0
    total_steps = 0
    total_dots = 0
    bad_episodes = 0

    for ep in range(episodes):
        state, _ = env.reset()
        initial_dots = env.dots_left
        done = False
        ep_reward = 0
        ep_steps = 0

        # 每回合重置防循环历史（如果启用防循环）
        if with_anti_cycle and hasattr(agent, 'action_history'):
            agent.action_history.clear()
            agent.cycle_count = 0

        if render:
            print(f"\n--- 第 {ep+1} 回合 初始状态 ---")
            env.render()

        while not done:
            # 根据模式选择动作
            if with_noise:
                action = agent.select_action_with_noise(state)
            else:
                action = agent.select_action_deterministic(state)

            next_state, reward, done, _ = env.step(action)
            ep_reward += reward
            ep_steps += 1
            state = next_state

            if render:
                print(f"步骤 {ep_steps}，动作 {action}，奖励 {reward:.2f}，当前总奖励 {ep_reward:.2f}")
                env.render()

        dots_eaten = initial_dots - env.dots_left
        print(f"Episode {ep+1}: Reward = {ep_reward:.2f}, Steps = {ep_steps}, Dots = {dots_eaten}")

        if ep_steps > 500:
            bad_episodes += 1

        total_reward += ep_reward
        total_steps += ep_steps
        total_dots += dots_eaten

    avg_reward = total_reward / episodes
    avg_steps = total_steps / episodes
    avg_dots = total_dots / episodes
    print("\n" + "=" * 50)
    print(f"演示总结（{mode_name}，{episodes} 回合）：")
    print(f"  Bad episodes (steps > 500): {bad_episodes} ({bad_episodes/episodes*100:.1f}%)")
    print(f"  Average reward : {avg_reward:.2f}")
    print(f"  Average steps  : {avg_steps:.2f}")
    print(f"  Average dots   : {avg_dots:.2f}")
    print("=" * 50)

In [ ]:
if __name__ == "__main__":
    # 初始化环境
    env = SimplePacmanEnv(grid_size=15)

    # 创建 agent（这里使用训练好的模型，请将 'model.pth' 替换为实际路径）
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path ='reward-model01.pth'
    agent = DQNAgent(action_size=4, device=device, model_path=model_path, anti_cycle_enabled=True)
    env=SimplePacmanEnv(grid_size=15)
    demonstrate(agent, env, 5, with_noise=False, with_anti_cycle=True, render=True)
    # demonstrate(agent, env, 5000, with_noise=False, with_anti_cycle=True, ·+++++++++++++++++++++++++++++
    # render=False)


========== 演示模式：确定性（无防循环） ==========


--- 第 1 回合 初始状态 ---
o o o o o . . . . o . . . o .
. . o o . . . . o o . o . . o
o o o o o . . . . o o . o o o
. o o o o o o o . o . o . o o
o o o . . o . o o o o o o . .
o o o . o o o . . . o . o . o
o . . o . o o . . o o . . o o
o . . o o o . . o o o o . o o
. . o . o o o . . o o . o o .
o . o o . . o . o . o o o o .
o o o o o o . o o . o . . . o
. o . o . . . o . o . o o o .
o o . o . o o o o o . o . . o
o . o o o o o . o . . o o o o
P o o . . o . o o o o o . . o
Steps: 0, Dots left: 135
步骤 1，动作 0，奖励 0.95，当前总奖励 0.95
o o o o o . . . . o . . . o .
. . o o . . . . o o . o . . o
o o o o o . . . . o o . o o o
. o o o o o o o . o . o . o o
o o o . . o . o o o o o o . .
o o o . o o o . . . o . o . o
o . . o . o o . . o o . . o o
o . . o o o . . o o o o . o o
. . o . o o o . . o o . o o .
o . o o . . o . o . o o o o .
o o o o o o . o o . o . . . o
. o . o . . . o . o . o o o .
o o . o . o o o o o . o . . o
o . o o o o o . o . . o o o o
. P o . . o . o 